# Zero-shot real-noise denoising with a one-step diffusion posterior mean

DMID (TPAMI 2024) holds the published state of the art among zero-shot methods on real noise: CC 37.99 dB, PolyU
38.62 dB, FMDD 33.40 dB. Before the frozen ADM 256×256 unconditional model denoises, a per-image network
optimisation (an ADMM loop of encoder–decoder training) turns the real noise into Gaussian-like noise of a known level.

This run asks whether pixel-shuffle downsampling (PD) can replace that optimisation. PD splits the image into s²
sub-images of pixels s apart, which breaks the spatial correlation demosaicing puts into camera noise. Each sub-image
is then denoised by the one-step posterior mean (Tweedie at the step matched to a wavelet noise estimate).

Measured, per image:

* noise statistics: residual std, lag-1/2 spatial correlation, σ̂ on the full image and on the PD sub-images;
* ADM at full resolution, step from m·σ̂ (σ̂ from the PD2 sub-images), over a grid of m;
* ADM after PD2 and PD4, step from m·σ̂ of each sub-image, over a grid of m;
* PD2 with the horizontal flip, PD2 refined by a full-resolution step at a lower level, and PD2 blended with full
  resolution;
* DRUNet (Gaussian, given σ̂) at full resolution and after PD2; Restormer trained on SIDD (supervised reference);
* DMID-d and DMID-p released outputs, rescored with the same code.

m is chosen on a development split (PolyU images 1–50) and reported on the rest (PolyU 51–100, CC, FMDD).
Kaggle: GPU T4 ×2, internet on, no attached inputs.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

In [ ]:
import hashlib, importlib.util, io, json, math, os, queue, re, subprocess, sys, threading, time, traceback, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from skimage.restoration import estimate_sigma

M_FULL = [0.5, 0.75, 1.0, 1.25, 1.5]      # full resolution: step from m · σ̂(PD2 sub-images)
M_PD = [0.8, 0.9, 1.0, 1.1, 1.25]         # PD: step from m · σ̂(each sub-image)
REFINE = [0.3, 0.5]                       # PD2 output re-denoised at full resolution at c · σ̂(PD2)
DEV_POLYU = 50                            # PolyU images 1..50 choose m; the rest of PolyU, CC and FMDD are test
TIME_LIMIT_H = 11.0
DEVICES = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())]
assert DEVICES, "GPU required"
torch.backends.cudnn.benchmark = False
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T_START, TIMES = time.time(), {}
print([str(d) for d in DEVICES], torch.__version__)


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if have and (size is None or have == size):
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {path.split('/')[-1]}: {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def natkey(s):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


t0 = time.time()
GH = "https://github.com/{}/releases/download/{}/{}"
FILES = {
    "adm": ("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt", 2_211_383_297),
    "datasets": (GH.format("Li-Tong-621/DMID", "v1.0", "Datasets.zip"), 367_018_356),
    "dmid_real": (GH.format("Li-Tong-621/DMID", "v1.0", "3-Real_world.zip"), 147_392_996),
    "restormer_real": (GH.format("swz30/Restormer", "v1.0", "real_denoising.pth"), 104_611_957),
    "drunet": (GH.format("cszn/KAIR", "v1.0", "drunet_color.pth"), 130_579_305),
}
SOURCES = {
    "cszn/KAIR": ("fc1732f4a4514e42ce15e5b3a1e18c828af47a1e", ["models/basicblock.py", "models/network_unet.py"]),
    "swz30/Restormer": ("68dc6ac472db26f16361150cb7a96a1bc87da93f", ["basicsr/models/archs/restormer_arch.py"]),
}
for repo, (commit, paths) in SOURCES.items():
    for path in paths:
        dest = f"{CACHE}/{repo.split('/')[1]}/{path}"
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        download(f"https://raw.githubusercontent.com/{repo}/{commit}/{path}", dest)
with ThreadPoolExecutor(5) as ex:
    P = dict(zip(FILES, ex.map(lambda kv: download(kv[1][0], f"{CACHE}/{kv[0]}{os.path.splitext(kv[1][0])[1]}", kv[1][1]), FILES.items())))
GD = f"{CACHE}/guided-diffusion"
if not os.path.isdir(GD):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/openai/guided-diffusion", GD], check=True)
TIMES["downloads"] = time.time() - t0
print(f"downloads {TIMES['downloads'] / 60:.1f} min")

## 1 · Data: DMID's release of CC (15), PolyU (100) and FMDD (48), all 512×512

In [ ]:
def read(z, name, mode="RGB"):
    return np.asarray(Image.open(io.BytesIO(z.read(name))).convert(mode))


PAIRS = {  # dataset: (clean folder, noisy folder, clean suffix, noisy suffix, grayscale)
    "CC": ("Datasets/Realworld/CC/GT/", "Datasets/Realworld/CC/Noisy/", "_mean", "_real", False),
    "PolyU": ("Datasets/Realworld/PolyU/PolyU_png/", "Datasets/Realworld/PolyU/Noisy/", "_mean", "_real", False),
    "FMDD": ("Datasets/Realworld/FMDD/gt/", "Datasets/Realworld/FMDD/raw/", "", "", True),
}
DATA = {}
with zipfile.ZipFile(P["datasets"]) as z:
    names = z.namelist()
    for ds, (cdir, ndir, csuf, nsuf, gray) in PAIRS.items():
        cleans = {os.path.splitext(n.split("/")[-1])[0].removesuffix(csuf): n for n in names if n.startswith(cdir) and not n.endswith("/")}
        noisys = {os.path.splitext(n.split("/")[-1])[0].removesuffix(nsuf): n for n in names if n.startswith(ndir) and not n.endswith("/")}
        stems = sorted(set(cleans) & set(noisys), key=natkey)
        mode = "L" if gray else "RGB"
        DATA[ds] = [(s, read(z, cleans[s], mode), read(z, noisys[s], mode)) for s in stems]
print({ds: len(v) for ds, v in DATA.items()}, {ds: v[0][1].shape for ds, v in DATA.items()})
SPLIT = {(ds, s): ("dev" if ds == "PolyU" and k < DEV_POLYU else "test") for ds, v in DATA.items() for k, (s, _, _) in enumerate(v)}

## 2 · Networks on every GPU (ADM fp16 with SDPA on contiguous q, k, v; Restormer real; DRUNet; LPIPS)

In [ ]:
sys.path[:0] = [GD]
from guided_diffusion.script_util import create_model
from guided_diffusion import unet as gd_unet


def sdpa_attention(self, qkv):
    bs, width, length = qkv.shape
    ch = width // (3 * self.n_heads)
    q, k, v = (z.transpose(1, 2).contiguous() for z in qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1))
    return F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(bs, -1, length)


gd_unet.QKVAttentionLegacy.forward = sdpa_attention
ADM_STATE = torch.load(P["adm"], map_location="cpu", weights_only=True)
for name in [m for m in sys.modules if m == "models" or m.startswith("models.")]:
    del sys.modules[name]
sys.path.insert(0, f"{CACHE}/KAIR")
from models.network_unet import UNetRes
sys.path.remove(f"{CACHE}/KAIR")
for name in [m for m in sys.modules if m == "models" or m.startswith("models.")]:
    del sys.modules[name]
RestormerNet = load_module("restormer_arch", f"{CACHE}/Restormer/basicsr/models/archs/restormer_arch.py").Restormer


def make_adm(device):
    m = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                     attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                     use_scale_shift_norm=True)
    m.load_state_dict(ADM_STATE)
    m.to(device).eval().requires_grad_(False).convert_to_fp16()
    return m


def make_restormer(device):
    m = RestormerNet(inp_channels=3, out_channels=3, dim=48, num_blocks=[4, 6, 6, 8], num_refinement_blocks=4, heads=[1, 2, 4, 8],
                     ffn_expansion_factor=2.66, bias=False, LayerNorm_type="BiasFree", dual_pixel_task=False)  # RealDenoising_Restormer.yml
    m.load_state_dict(torch.load(P["restormer_real"], map_location="cpu", weights_only=True)["params"], strict=True)
    return m.to(device).eval().requires_grad_(False)


def make_drunet(device):
    m = UNetRes(in_nc=4, out_nc=3, nc=[64, 128, 256, 512], nb=4, act_mode="R", downsample_mode="strideconv",
                upsample_mode="convtranspose", bias=False)
    m.load_state_dict(torch.load(P["drunet"], map_location="cpu", weights_only=True), strict=True)
    return m.to(device).eval().requires_grad_(False)


import lpips

CTX = [dict(dev=d, adm=make_adm(d), restormer=make_restormer(d), drunet=make_drunet(d),
            lpips=lpips.LPIPS(net="alex", verbose=False).to(d)) for d in DEVICES]
ADM_STATE.clear()
print("networks ready")

## 3 · Estimators

In [ ]:
BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)
NEG_LOG_ABAR = (-torch.log(torch.cumprod(1 - BETAS, 0))).numpy()


def matched(sigma01):
    """Continuous step t and abar with abar = 1 / (1 + (2σ)^2)."""
    return float(np.interp(math.log1p(4 * sigma01 ** 2), NEG_LOG_ABAR, np.arange(1000))), 1 / (1 + 4 * sigma01 ** 2)


@torch.no_grad()
def tweedie(model, y, sigmas):
    """One-step posterior means of y [B,3,H,W] (unclipped, [0,1] units), sample b embedded at the step for sigmas[b]."""
    steps = [matched(float(s)) for s in sigmas]
    t = torch.tensor([a for a, _ in steps], dtype=torch.float32, device=y.device)
    ab = torch.tensor([b for _, b in steps], dtype=torch.float32, device=y.device).view(-1, 1, 1, 1)
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 64, 0, (-h) % 64), mode="reflect") if (h % 64 or w % 64) else y
    x = ab.sqrt() * (2 * yp - 1)
    eps = model(x, t)[:, :3].float()
    return (((x - (1 - ab).sqrt() * eps) / ab.sqrt() + 1) / 2)[..., :h, :w]


def pd_split(y, s):
    """[1,3,H,W] -> [s*s,3,H/s,W/s], sub-image k = (k // s, k % s) offset."""
    return torch.cat([y[..., i::s, j::s] for i in range(s) for j in range(s)])


def pd_merge(subs, s):
    k, c, h, w = subs.shape
    out = torch.zeros(1, c, h * s, w * s, device=subs.device, dtype=subs.dtype)
    for n in range(k):
        out[..., n // s::s, n % s::s] = subs[n]
    return out


def sigma_hat(y):
    """Wavelet MAD noise estimate (skimage), averaged over channels; y [3,H,W] on any device."""
    return float(estimate_sigma(y.permute(1, 2, 0).double().cpu().numpy(), channel_axis=-1, average_sigmas=True))


@torch.no_grad()
def run_restormer(net, y):
    h, w = y.shape[-2:]
    return net(F.pad(y, (0, (-w) % 8, 0, (-h) % 8), mode="reflect"))[..., :h, :w]


@torch.no_grad()
def run_drunet(net, y, sigmas):
    h, w = y.shape[-2:]
    yp = F.pad(y, (0, (-w) % 8, 0, (-h) % 8), mode="replicate")
    level = torch.tensor(sigmas, dtype=yp.dtype, device=yp.device).view(-1, 1, 1, 1).expand(-1, 1, *yp.shape[-2:])
    return net(torch.cat([yp, level], 1))[..., :h, :w]


_g = torch.exp(-torch.arange(-5, 6, dtype=torch.float64) ** 2 / (2 * 1.5 ** 2))
SSIM_WINDOW = (_g / _g.sum())[:, None] * (_g / _g.sum())[None, :]


def ssim(a, b):
    """skimage structural_similarity(gaussian_weights=True, sigma=1.5, use_sample_covariance=False, data_range=255)."""
    w = SSIM_WINDOW.to(a.device)[None, None].repeat(a.shape[0], 1, 1, 1)
    conv = lambda x: F.conv2d(x[None], w, groups=a.shape[0])[0]
    ux, uy = conv(a), conv(b)
    vx, vy, vxy = conv(a * a) - ux * ux, conv(b * b) - uy * uy, conv(a * b) - ux * uy
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    return float((((2 * ux * uy + c1) * (2 * vxy + c2)) / ((ux * ux + uy * uy + c1) * (vx + vy + c2))).mean())


def to_u8(x, gray):
    """[1,3,H,W] float -> uint8 [C,H,W]; grayscale sets average the three channels first, as DMID does for FMDD."""
    x = x[0].clamp(0, 1)
    if gray:
        x = x.mean(0, keepdim=True)
    return (x * 255).round()


def scores(out_u8, ref_u8, lp):
    a, b = out_u8.double(), ref_u8.double()
    mse = (a - b).pow(2).mean().item()
    a3, b3 = (a.expand(3, -1, -1), b.expand(3, -1, -1)) if a.shape[0] == 1 else (a, b)
    with torch.no_grad():
        lpv = lp(a3.float()[None] / 127.5 - 1, b3.float()[None] / 127.5 - 1).item()
    return dict(psnr=10 * math.log10(255 ** 2 / mse) if mse > 0 else float("inf"), ssim=ssim(a, b), lpips=lpv)

## 4 · Every estimator on every image (each method guarded: one failure does not stop the others)

In [ ]:
def run_image(ctx, job):
    ds, k = job
    stem, clean, noisy = DATA[ds][k]
    gray = clean.ndim == 2
    dev, adm = ctx["dev"], ctx["adm"]
    to3 = lambda a: torch.from_numpy(np.repeat(a[..., None], 3, -1) if a.ndim == 2 else a).permute(2, 0, 1).float()
    y = (to3(noisy) / 255)[None].to(dev)
    ref = torch.from_numpy(clean[None] if gray else clean.transpose(2, 0, 1).copy()).to(dev).float()
    resid = (to3(noisy) - to3(clean)) / 255
    lag = {d: float(np.corrcoef(resid[..., :-d].flatten().numpy(), resid[..., d:].flatten().numpy())[0, 1]) for d in (1, 2)}
    pd2, pd4 = pd_split(y, 2), pd_split(y, 4)
    s_full, s_pd2, s_pd4 = sigma_hat(y[0]), [sigma_hat(s) for s in pd2], [sigma_hat(s) for s in pd4]
    s2 = float(np.mean(s_pd2))
    stats = dict(dataset=ds, image=stem, split=SPLIT[(ds, stem)], resid_std=float(resid.std()), corr_lag1=lag[1], corr_lag2=lag[2],
                 sigma_full=s_full, sigma_pd2=s2, sigma_pd4=float(np.mean(s_pd4)))
    outs, errors = {"noisy input": y}, []

    def guard(name, fn):
        try:
            outs.update(fn())
        except Exception:
            errors.append((name, traceback.format_exc(limit=3)))

    guard("full", lambda: {f"ADM full, m={m:g}": tweedie(adm, y, [m * s2]) for m in M_FULL})
    guard("pd2", lambda: {f"ADM PD2, m={m:g}": pd_merge(tweedie(adm, pd2, [m * s for s in s_pd2]), 2) for m in M_PD})
    guard("pd4", lambda: {f"ADM PD4, m={m:g}": pd_merge(tweedie(adm, pd4, [m * s for s in s_pd4]), 4) for m in M_PD})

    def flip():
        f = pd_merge(tweedie(adm, pd_split(y.flip(-1), 2), [sigma_hat(s) for s in pd_split(y.flip(-1), 2)]), 2).flip(-1)
        return {"ADM PD2 ×2 (flip), m=1": (outs["ADM PD2, m=1"] + f) / 2}

    guard("flip", flip)
    guard("refine", lambda: {f"ADM PD2 + full-res step at {c:g}σ̂": tweedie(adm, outs["ADM PD2, m=1"], [c * s2]) for c in REFINE})
    guard("blend", lambda: {"ADM (PD2 + full m=1) / 2": (outs["ADM PD2, m=1"] + outs["ADM full, m=1"]) / 2})
    guard("restormer", lambda: {"Restormer (SIDD-trained)": run_restormer(ctx["restormer"], y)})
    guard("drunet", lambda: {"DRUNet full, σ̂(PD2)": run_drunet(ctx["drunet"], y, [s2]),
                             "DRUNet PD2": pd_merge(run_drunet(ctx["drunet"], pd2, s_pd2), 2)})
    rows = [dict(dataset=ds, image=stem, split=stats["split"], method=m, **scores(to_u8(o, gray), ref, ctx["lpips"])) for m, o in outs.items()]
    return rows, stats, errors


def worker_pool(jobs, fn):
    """Run fn(ctx, job) over jobs with one thread per GPU; returns (results, failures)."""
    q, results, failures, lock = queue.Queue(), [], [], threading.Lock()
    for j in jobs:
        q.put(j)

    def loop(ctx):
        while time.time() - T_START < TIME_LIMIT_H * 3600:
            try:
                job = q.get_nowait()
            except queue.Empty:
                return
            try:
                out = fn(ctx, job)
                with lock:
                    results.append(out)
            except Exception:
                with lock:
                    failures.append((job, traceback.format_exc()))

    threads = [threading.Thread(target=loop, args=(c,)) for c in CTX]
    for th in threads:
        th.start()
    t0, last = time.time(), time.time()
    while any(th.is_alive() for th in threads):
        time.sleep(20)
        if time.time() - last > 600:
            last = time.time()
            print(f"  {len(results)}/{len(jobs)} done, {len(failures)} failed, {(time.time() - t0) / 60:.1f} min", flush=True)
    for job, tb in failures[:3]:
        print("FAILED", job, tb)
    return results, failures


t0 = time.time()
jobs = [(ds, k) for ds in ("CC", "FMDD", "PolyU") for k in range(len(DATA[ds]))]
res, fail = worker_pool(jobs[:3], run_image)   # smoke test: first image of CC before the full run
print("smoke test:", len(res), "ok,", len(fail), "failed;", [e for _, _, errs in res for e in errs][:2])
res2, fail2 = worker_pool(jobs[3:], run_image)
res, fail = res + res2, fail + fail2
SCORES = pd.DataFrame([r for rows, _, _ in res for r in rows])
STATS = pd.DataFrame([s for _, s, _ in res])
METHOD_ERRORS = [(s["dataset"], s["image"], name, tb) for _, s, errs in res for name, tb in errs]
SCORES.to_csv(f"{OUT}/scores.csv", index=False)
STATS.to_csv(f"{OUT}/noise_stats.csv", index=False)
TIMES["denoising"] = time.time() - t0
print(f"{len(res)}/{len(jobs)} images, {len(fail)} failed images, {len(METHOD_ERRORS)} failed methods, {TIMES['denoising'] / 60:.1f} min")
for e in METHOD_ERRORS[:3]:
    print("METHOD ERROR", e)

## 5 · DMID's released real-noise outputs, rescored with the same code (matched to our clean images by content)

In [ ]:
key = lambda a: hashlib.sha1(np.ascontiguousarray(a).tobytes()).hexdigest()


def as_tensor(a):
    """uint8 [H,W] or [H,W,3] -> float [C,H,W] on the first GPU."""
    return torch.from_numpy(a[None] if a.ndim == 2 else a.transpose(2, 0, 1).copy()).float().to(DEVICES[0])


CLEAN_KEYS = {key(c): (ds, s) for ds, v in DATA.items() for s, c, _ in v}
dmid_rows = []
lp0 = CTX[0]["lpips"]
with zipfile.ZipFile(P["dmid_real"]) as z:
    for n in z.namelist():
        m = re.match(r"3-Real_world/(CC|PolyU|FMDD)-([dp])/denoise/(\d+)_denoise\.png$", n)
        if not m:
            continue
        ds, variant, idx = m.groups()
        gray = ds == "FMDD"
        clean = read(z, f"3-Real_world/{ds}-{variant}/clean/{idx}_clean.png", "L" if gray else "RGB")
        den = read(z, n, "L" if gray else "RGB")
        match = CLEAN_KEYS.get(key(clean))
        dmid_rows.append(dict(dataset=ds, image=match[1] if match else f"unmatched {idx}", split=SPLIT.get(match, "test"),
                              method=f"DMID-{variant} (released)", **scores(as_tensor(den), as_tensor(clean), lp0)))
DMID = pd.DataFrame(dmid_rows)
DMID.to_csv(f"{OUT}/dmid_released_scores.csv", index=False)
print(DMID.groupby(["dataset", "method"])[["psnr", "ssim", "lpips"]].mean().round(4).to_string())
print("unmatched DMID clean images:", int(DMID.image.str.startswith("unmatched").sum()))

## 6 · Results

In [ ]:
ALL = pd.concat([SCORES, DMID], ignore_index=True)
ALL.to_csv(f"{OUT}/all_scores.csv", index=False)
mean = ALL.groupby(["method", "dataset"])[["psnr", "ssim", "lpips"]].mean()
for metric in ("psnr", "ssim", "lpips"):
    print(f"\nmean {metric.upper()} (all images)")
    print(mean[metric].unstack().round(4 if metric != "psnr" else 3).to_string())
dev = SCORES[SCORES.split == "dev"].groupby("method").psnr.mean()
chosen = {fam: dev[[m for m in dev.index if m.startswith(fam)]].idxmax() for fam in ("ADM full, m=", "ADM PD2, m=", "ADM PD4, m=")}
print("\nchosen on PolyU 1-50 (dev):", chosen)
test = ALL[ALL.split == "test"]
keep = list(chosen.values()) + [m for m in ALL.method.unique() if not any(m.startswith(f) for f in chosen)]
print("\ntest split (PolyU 51-100, CC, FMDD), mean PSNR / SSIM / LPIPS")
print(test[test.method.isin(keep)].groupby(["method", "dataset"])[["psnr", "ssim", "lpips"]].mean().round(4).unstack().to_string())
print("\nnoise statistics (means)\n", STATS.groupby("dataset")[["resid_std", "corr_lag1", "corr_lag2", "sigma_full", "sigma_pd2", "sigma_pd4"]].mean().round(4).to_string())
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(minutes={k: v / 60 for k, v in TIMES.items()}, chosen_on_dev=chosen, m_full=M_FULL, m_pd=M_PD, refine=REFINE,
                   failed_images=[j for j, _ in fail], failed_methods=[e[:3] for e in METHOD_ERRORS]), f, indent=1)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)), f"total {(time.time() - T_START) / 60:.1f} min")